# CODLING Training on Google Colab
==================================

This notebook demonstrates training CODLING on Colab with T4 GPU.

**Contents:**
- Load/process training data (The Pile or code data)
- Set up tokenizer (GPT-2 or custom)
- Configure trainer for T4 GPU
- Run training loop with metrics
- Save checkpoints to Google Drive
- Resume from checkpoint
- Training tips for Colab

---

## 1. Setup and Configuration
Initialize the environment and set up paths.

In [ ]:
import sys
sys.path.insert(0, '/content')

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import gc

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
CHECKPOINT_DIR = '/content/drive/MyDrive/codling_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 2. Load Tokenizer
Load GPT-2 tokenizer (or custom tokenizer for CODLING).

In [ ]:
print("=" * 50)
print("LOADING TOKENIZER")
print("=" * 50)

# Use GPT-2 tokenizer as starting point
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Add special tokens
special_tokens = {
    "pad_token": "<|pad|>",
    "bos_token": "<|bos|>",
    "eos_token": "<|eos|>",
}
tokenizer.add_special_tokens(special_tokens)

print(f"Vocabulary size: {len(tokenizer)}")
print(f"Bos token: {tokenizer.bos_token} (id: {tokenizer.bos_token_id})")
print(f"Eos token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
print(f"Pad token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")

# Test encoding
test_text = "def hello_world():\n    print('Hello, World!')"
encoded = tokenizer(test_text, return_tensors="pt")
decoded = tokenizer.decode(encoded["input_ids"][0])
print(f"\nTest encoding:\n  Input: {test_text}")
print(f"  IDs: {encoded['input_ids']}")
print(f"  Decoded: {decoded}")

## 3. Load Training Data
Load and preprocess training data from The Pile or code datasets.

In [ ]:
print("=" * 50)
print("LOADING TRAINING DATA")
print("=" * 50)

# Option 1: Use a small sample for demo (replace with full dataset)
# from datasets import load_dataset
# dataset = load_dataset("the_pile", split="train[:10%]")  # First 10%

# Option 2: Create synthetic code dataset for demo
def generate_synthetic_code_data(n_samples=10000):
    """Generate synthetic code data for demonstration."""
    
    code_templates = [
        "def function_{i}(x):\n    return x * {i}\n",
        "class Class{i}:\n    def __init__(self):\n        self.value = {i}\n",
        "def calculate_{i}(a, b):\n    return a + b * {i}\n",
        "for i in range({i}):\n    print(i)\n",
        "if x > {i}:\n    return True\nelse:\n    return False\n",
    ]
    
    data = []
    for i in range(n_samples):
        template = code_templates[i % len(code_templates)]
        text = template.format(i=i%100)
        data.append({"text": text})
    
    return data

print("Generating synthetic code dataset...")
train_data = generate_synthetic_code_data(n_samples=5000)
val_data = generate_synthetic_code_data(n_samples=500)

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"\nSample text:\n{train_data[0]['text']}")

## 4. Dataset Class
Create PyTorch dataset for training.

In [ ]:
class CodeDataset(Dataset):
    """Dataset for code text."""
    
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        
        # Tokenize with padding and truncation
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        
        input_ids = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()
        
        # Labels are the same as input_ids for causal LM
        labels = input_ids.clone()
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

# Create datasets
MAX_LENGTH = 256  # Shorter for Colab demo
train_dataset = CodeDataset(train_data, tokenizer, max_length=MAX_LENGTH)
val_dataset = CodeDataset(val_data, tokenizer, max_length=MAX_LENGTH)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

# Test sample
sample = train_dataset[0]
print(f"\nSample shapes:")
print(f"  input_ids: {sample['input_ids'].shape}")
print(f"  attention_mask: {sample['attention_mask'].shape}")
print(f"  labels: {sample['labels'].shape}")

## 5. Model Configuration
Define model configuration for training.

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    """Training configuration."""
    # Model
    vocab_size: int = 50257
    hidden_size: int = 256
    num_layers: int = 4
    num_heads: int = 4
    intermediate_size: int = 1024
    max_position_embeddings: int = 512
    d_state: int = 64
    
    # Training
    batch_size: int = 8
    learning_rate: float = 1e-4
    num_epochs: int = 3
    warmup_steps: int = 100
    weight_decay: float = 0.01
    gradient_accumulation_steps: int = 4
    max_grad_norm: float = 1.0
    
    # Logging
    log_interval: int = 10
    save_interval: int = 100
    eval_interval: int = 50

config = TrainingConfig()

print("=" * 50)
print("TRAINING CONFIGURATION")
print("=" * 50)
for key, value in config.__dict__.items():
    print(f"  {key}: {value}")

## 6. Build Model
Create the model architecture for training.

In [ ]:
import torch.nn.functional as F

# Simplified SSM Block (same as architecture notebook)
class SimplifiedSSMBlock(nn.Module):
    """Simplified SSM block (Mamba-style)."""
    
    def __init__(self, hidden_size: int, d_state: int = 64):
        super().__init__()
        self.hidden_size = hidden_size
        self.d_state = d_state
        self.expand = 2
        
        self.in_proj = nn.Linear(hidden_size, hidden_size * self.expand * 2, bias=False)
        self.x_proj = nn.Linear(hidden_size * self.expand, d_state, bias=False)
        self.dt_proj = nn.Linear(d_state, hidden_size * self.expand, bias=True)
        self.A_log = nn.Parameter(torch.randn(d_state, hidden_size * self.expand))
        self.D = nn.Parameter(torch.ones(hidden_size * self.expand))
        self.out_proj = nn.Linear(hidden_size * self.expand, hidden_size, bias=False)
        self.norm = nn.RMSNorm(hidden_size)
        
    def forward(self, x):
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x_inner, z = xz.chunk(2, dim=-1)
        x_conv = F.silu(x_inner)
        ssm_params = self.x_proj(x_conv)
        dt = self.dt_proj(ssm_params)
        dt = F.softplus(dt)
        A = -torch.exp(self.A_log.float())
        y = x_conv * torch.sigmoid(dt) + self.D * x_conv
        y = y * F.silu(z)
        out = self.out_proj(y)
        return out + residual


class SimplifiedCodlingLM(nn.Module):
    """Simplified CODLING for causal language modeling."""
    
    def __init__(self, vocab_size: int, hidden_size: int, num_layers: int, 
                 max_position_embeddings: int, d_state: int):
        super().__init__()
        
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size)
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        
        self.layers = nn.ModuleList([
            SimplifiedSSMBlock(hidden_size, d_state)
            for _ in range(num_layers)
        ])
        
        self.final_norm = nn.RMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.token_embeddings.weight
        
    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape
        
        hidden_states = self.token_embeddings(input_ids)
        
        position_ids = torch.arange(seq_len, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)
        hidden_states = hidden_states + self.position_embeddings(position_ids)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        
        hidden_states = self.final_norm(hidden_states)
        logits = self.lm_head(hidden_states)
        
        return logits

# Create model
model = SimplifiedCodlingLM(
    vocab_size=len(tokenizer),
    hidden_size=config.hidden_size,
    num_layers=config.num_layers,
    max_position_embeddings=config.max_position_embeddings,
    d_state=config.d_state,
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 50)
print("MODEL CREATED")
print("=" * 50)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 7. Optimizer and Scheduler
Set up optimizer, scheduler, and training components.

In [ ]:
# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
    betas=(0.9, 0.95),
    eps=1e-8,
)

# Learning rate scheduler
total_steps = len(train_dataset) * config.num_epochs // config.batch_size
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=config.warmup_steps,
    num_training_steps=total_steps,
)

# Loss function
loss_fn = nn.CrossEntropyLoss()

# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=0,
)

print("=" * 50)
print("OPTIMIZER AND SCHEDULER SETUP")
print("=" * 50)
print(f"Optimizer: AdamW")
print(f"Learning rate: {config.learning_rate}")
print(f"Weight decay: {config.weight_decay}")
print(f"Total steps: {total_steps}")
print(f"Warmup steps: {config.warmup_steps}")
print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 8. Training Loop
Run the main training loop with metrics tracking.

In [ ]:
def train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device, config):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc="Training")
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        
        logits = model(input_ids)
        
        # Shift for causal LM
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        loss = loss_fn(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )
        
        # Scale loss for gradient accumulation
        loss = loss / config.gradient_accumulation_steps
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        if config.max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        
        # Optimizer step
        if (step + 1) % config.gradient_accumulation_steps == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * config.gradient_accumulation_steps
        
        # Progress bar update
        progress_bar.set_postfix({
            "loss": f"{loss.item() * config.gradient_accumulation_steps:.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}",
        })
        
        # Memory usage
        if torch.cuda.is_available():
            mem = torch.cuda.memory_allocated() / 1024**2
            progress_bar.set_postfix({
                "loss": f"{loss.item() * config.gradient_accumulation_steps:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                "mem": f"{mem:.0f}MB",
            })
    
    return total_loss / len(train_loader)


def evaluate(model, val_loader, loss_fn, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            
            logits = model(input_ids)
            
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            
            loss = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )
            
            total_loss += loss.item()
    
    return total_loss / len(val_loader)

print("Training functions defined!")

## 9. Run Training
Execute the training loop and save checkpoints.

In [ ]:
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(config.num_epochs):
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch + 1}/{config.num_epochs}")
    print(f"{'='*60}")
    
    # Training
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device, config)
    train_losses.append(train_loss)
    
    # Validation
    val_loss = evaluate(model, val_loader, loss_fn, device)
    val_losses.append(val_loss)
    
    print(f"\n📊 Epoch {epoch + 1} Summary:")
    print(f"   Train Loss: {train_loss:.4f}")
    print(f"   Val Loss:   {val_loss:.4f}")
    print(f"   LR:         {scheduler.get_last_lr()[0]:.2e}")
    
    # Save checkpoint
    checkpoint_path = f"{CHECKPOINT_DIR}/checkpoint-epoch-{epoch+1}.pt"
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
    }, checkpoint_path)
    
    print(f"\n✅ Checkpoint saved: {checkpoint_path}")
    
    # Track best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_checkpoint_path = f"{CHECKPOINT_DIR}/best_model.pt"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
        }, best_checkpoint_path)
        print(f"✅ Best model saved: {best_checkpoint_path}")
    
    # Clear cache
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)

## 10. Plot Training Curves
Visualize training and validation loss.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses)+1), train_losses, 'b-o', label='Train Loss')
plt.plot(range(1, len(val_losses)+1), val_losses, 'r-o', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/training_curves.png', dpi=150)
plt.show()

print(f"Training curves saved to {CHECKPOINT_DIR}/training_curves.png")

## 11. Resume from Checkpoint
Load a saved checkpoint to continue training.

In [ ]:
print("=" * 50)
print("RESUME FROM CHECKPOINT")
print("=" * 50)

import os
checkpoint_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')]
print(f"\n📁 Available checkpoints in {CHECKPOINT_DIR}:")
for f in checkpoint_files:
    size_mb = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / (1024**2)
    print(f"   • {f} ({size_mb:.2f} MB)")

# Function to load checkpoint
def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    """Load a checkpoint and resume training."""
    print(f"\n📥 Loading checkpoint: {checkpoint_path}")
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load optimizer
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    epoch = checkpoint.get('epoch', 0)
    print(f"✅ Checkpoint loaded, resuming from epoch {epoch+1}")
    
    return model, optimizer, scheduler, epoch

# Example: How to resume (uncomment to use)
# model, optimizer, scheduler, start_epoch = load_checkpoint(
#     model, optimizer, scheduler, 
#     f"{CHECKPOINT_DIR}/checkpoint-epoch-1.pt"
# )

## 12. Colab Tips
Important tips for training on Google Colab.

In [ ]:
print("=" * 60)
print("COLAB TRAINING TIPS")
print("=" * 60)

print("""
⚠️  IMPORTANT TIPS FOR COLAB TRAINING:

1. PREVENT DISCONNECTION:
   - Keep the browser tab open
   - Use Chrome extension "Colab Auto Reconnect"
   - Run this JS in console to prevent timeout:
     function ClickConnect(){
       console.log("Working");
       document.querySelector("#colab-connect-button").click()
     }
     setInterval(ClickConnect, 60000)

2. SAVE CHECKPOINTS FREQUENTLY:
   - Save every epoch to Google Drive
   - Keep local backup too
   - Use smaller batch sizes to save more often

3. GPU MEMORY MANAGEMENT:
   - Use gradient accumulation for larger effective batch
   - Clear cache: torch.cuda.empty_cache()
   - Monitor: !nvidia-smi

4. BATTERY OPTIMIZATION:
   - Connect to AC power
   - Set "GPU" not "T4" in notebook settings
   - Use "High RAM" runtime if available

5. RESUMING AFTER DISCONNECTION:
   - Save checkpoint before each epoch
   - Load checkpoint and continue
   - May need to reinitialize model and optimizer

6. MONITORING:
   - Use tensorboard or wandb
   - Check GPU with: !nvidia-smi
   - Monitor memory with: torch.cuda.memory_allocated()
""")

## 13. Summary
Training complete.

In [ ]:
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)

print(f"""
✅ Training completed!

Checkpoints saved to: {CHECKPOINT_DIR}

Files created:
  - checkpoint-epoch-*.pt (model checkpoints)
  - best_model.pt (best validation loss)
  - training_curves.png (loss visualization)

Next steps:
  1. Open 04_evaluation.ipynb to evaluate on HumanEval
  2. Try larger models and datasets
  3. Experiment with different hyperparameters
""")